# Connection

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from openpyxl.utils.datetime import to_excel
from pandas import read_excel
from sympy import false
from functions_POC import *
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
import pandas as pd
import time

In [ ]:
client = AzureOpenAI(
    azure_endpoint="https://litellm.ai.viadee.cloud",
    #api_key=os.environ["OPENAI_API_KEY"],
    api_key = os.getenv("OPENAI_API_KEY"),
    api_version="",
)

In [ ]:
# DATA created by claude #1
document = [
     f"""This Directive establishes a unified framework for the regulation of digital payment services and the safeguarding of consumer data within the jurisdiction of the Federal Commerce Authority.""",
    "The provisions herein apply to all licensed payment service providers, payment facilitators, and affiliated merchants operating within the regulated marketplace.",
    "The Federal Commerce Authority recognizes the growing importance of digital commerce and the need to balance innovation with consumer protection.",
f"""For the purposes of this Directive, 'payment service provider' refers to any entity authorized to initiate, process, or settle electronic payment transactions on behalf of consumers or merchants.""",
"'Consumer data' shall mean any personally identifiable information, financial account details, transaction histories, or behavioral data collected during the provision of payment services.",
"'Sensitive authentication data' includes, but is not limited to, full card numbers, security codes, PINs, and biometric identifiers used for transaction authorization.",
"All payment service providers shall conduct a comprehensive identity verification of each merchant prior to granting access to payment processing services.",
"The identity verification process must include validation of the merchant's legal registration, beneficial ownership structure, and principal place of business.",
"Payment service providers are prohibited from onboarding any merchant that appears on a government-maintained sanctions or exclusion list.",
"Where a merchant operates in a high-risk category as defined in Annex II, the payment service provider shall apply enhanced due diligence measures appropriate to the level of risk.",
"Verification records shall be retained for a minimum period of seven years from the date of merchant account closure.",
"This chapter outlines the obligations of payment service providers with respect to consumer data collected during payment processing.",
"Payment service providers shall implement adequate technical and organizational safeguards to protect consumer data against unauthorized access, disclosure, or destruction.",
"Consumer data must not be shared with third parties for marketing purposes without the explicit, documented consent of the consumer.",
"All consumer data transfers across jurisdictional boundaries require prior authorization from the Data Oversight Board and must employ end-to-end encryption.",
"In the event of a data breach affecting consumer records, the payment service provider must notify the Federal Commerce Authority within 72 hours of becoming aware of the breach.",
"Payment service providers shall establish and maintain a real-time transaction monitoring system capable of detecting anomalous or potentially fraudulent activity.",
"Transactions exceeding the threshold amount specified in Annex III must be flagged for manual review and reported to the Financial Intelligence Unit within five business days.",
"The Federal Commerce Authority may issue supplementary guidance on monitoring methodologies from time to time.",
"Every payment service provider must establish a formal consumer complaint resolution procedure that is easily accessible and clearly communicated to all users.",
"Consumer complaints must be acknowledged within two business days and resolved within 30 calendar days of receipt.",
f"""If a complaint cannot be resolved within the prescribed timeframe, the provider shall escalate the matter to the Consumer Mediation Office and inform the consumer in writing of the escalation.""",
"This chapter addresses the requirements for maintaining operational resilience in the delivery of payment services.",
"Payment service providers operating critical payment infrastructure shall conduct annual business continuity testing and submit results to the Federal Commerce Authority.",
"Non-compliance with the provisions of this Directive may result in administrative penalties, license suspension, or revocation, as determined by the Federal Commerce Authority.",
"The penalty framework established under this Directive does not preclude additional civil or criminal liability under applicable law.",
"Beneficial ownership verification should be conducted in a manner that is proportionate to the overall risk profile of the merchant, taking into account factors that the compliance team considers relevant at the time of assessment.",
"Transaction monitoring alerts must be reviewed and actioned within a timeframe that is reasonable given the nature and severity of the alert, and in accordance with generally accepted industry practices.",
"Staff supporting merchant accounts must receive adequate and up-to-date compliance training that is sufficient to enable them to perform their duties competently."



]

process =  f"""
The merchant submits an online application form including business name, registration number, contact information, and requested payment services; An automated system checks the form for completeness and generates a case ID; Incomplete applications are returned to the merchant with specific instructions on missing fields.
A compliance analyst verifies the merchant's legal registration against the national business registry; The analyst also confirms the merchant's principal place of business through address validation services; Beneficial ownership verification is performed only when flagged by the automated risk scoring tool, not for all merchants.
The merchant's legal entity name and all beneficial owners are screened against all government-maintained sanctions and exclusion lists; Any match results in automatic rejection of the application; Screening results are logged and attached to the case file.
The merchant is assigned a risk category (low, medium, or high) based on its business type, geography, and transaction volume projections; High-risk merchants receive a general advisory notice but no additional due diligence procedures are applied beyond the standard verification; The risk category is recorded in the merchant profile.
Technical staff configure the merchant's payment processing environment with encryption, access controls, and tokenization of sensitive authentication data; All consumer data stored in the system is protected using AES-256 encryption at rest and TLS 1.3 in transit; A data protection impact assessment is completed and filed.
The merchant's point-of-sale or e-commerce system is integrated with the payment gateway; Technical staff run a series of test transactions to validate connectivity, latency, and error handling; Test results are documented and approved by a QA lead.
The merchant's account is enrolled in the batch-processed transaction monitoring system, which runs nightly scans to detect patterns indicative of fraud; Alerts are reviewed by the fraud operations team on the following business day; The system does not operate in real time.
The merchant reviews and signs the standard service agreement, which includes data handling terms, fee schedules, and dispute resolution clauses; The agreement includes a clause authorizing the provider to share anonymized transaction data with analytics partners for market research; The executed agreement is stored in the contract management system.
New team members supporting the merchant account attend a training session covering operational procedures, escalation paths, and compliance awareness; The training is delivered quarterly and covers general regulatory topics; Attendance records are maintained by HR.
Following successful completion of all prior steps, the merchant account is activated in the production environment; The merchant receives onboarding confirmation with login credentials and support contact details; A 30-day post-activation review is scheduled.
"""

# Classification of the document chunks

In [ ]:
### Comment: The prompt is designed to guide the LLM in classifying document chunks as containing compliance rules or not, and if so, what type of rule. The instructions are clear and provide specific criteria for classification, which should help improve the accuracy of the LLM's responses.

classification_prompt_template = '''
You are an expert compliance analyst. Determine whether the document excerpt below contains a compliance rule or requirement.

Document chunk:
"""
{chunk}
"""

A compliance rule is a statement that prescribes what MUST be done, SHOULD be done, or MUST NOT be done or SHOULD NOT be done. It includes:
-Statements that describe a commitment or promise to take a specific action in response to a trigger or condition, even if they do not use explicit obligation language, can also be considered compliance rules if they create an expectation of performance.
-Statements how the action should be performed
- Obligations (must, shall, required to, have to, will)
- Prohibitions (must not, shall not, cannot, prohibited, will not)
- Timeframes/deadlines (within X days, at least every X days)
- Required actions or procedures
- Conditions that trigger specific actions



NOT a compliance rule:
- Scope statements (e.g., "This applies to X only")
- General descriptions (e.g., "The process involves...")
- Explanatory text without requirements
- Background information

Answer in JSON only with these fields exactly:
{{
  "reasoning": "One- or two-sentence explanation",
  "rule_type": "obligation" | "prohibition" | "timeframe" | "procedure" | "conditional" | "none" ,
  "contains_compliance_rule": "yes" | "no",
}}
'''
#Anmerkung/Frage: kein rule_type 'sonstiges'??
###MEISTENS FUNKTIONIERT GUT; ABER HÄUFIG VARIABLE ANTWORTEN bei MANCHEN _ MAX OF 3 audwählen? ACHTUNG; ich habe die regeln und definitionen beschrieben, wenn es neue kommen, dann kann es falsch sein!!! quasi overfitting

In [ ]:
timings = {}
start = time.time()

df_chunk_classification = run_step(
    document, process, client, classification_prompt_template,
    step_name="0.0_chunk_classification")

timings["chunk_classification"] = time.time() - start

In [ ]:
timings

In [ ]:
df_chunk_classification

#ERKENNTNIS: Sätze wie "When we receive your Complaint, we will acknowledge that we have received it." werden nicht als compliance rules erkannt, obwohl sie eigentlich eine Verpflichtung enthalten. Möglicherweise muss die Definition von "compliance rule" im Prompt angepasst werden, damit auch solche Sätze erfasst werden. Zum Beispiel könnte man die Definition erweitern um: "Statements that describe a commitment or promise to take a specific action in response to a trigger or condition, even if they do not use explicit obligation language, can also be considered compliance rules if they create an expectation of performance."

In [ ]:
compliance_rules = [
    (row['chunk_id'], row['chunk_text'])
    for _, row in df_chunk_classification.iterrows()
    if row['contains_compliance_rule'] != 'no' #or row['contains_compliance_rule'] == 'no'
]
chunk_ids, chunk_texts = zip(*compliance_rules)

# convert to lists if needed
chunk_ids = list(chunk_ids)
compliance_rule_chunks  = list(chunk_texts)

# Relevance Classification

In [ ]:
#Plan B Prompt, zeigte weniger 'NO'
document_evaluation_prompt = """
# Instructions

You are an expert in regulatory compliance analysis. Analyze whether a regulatory text passage is relevant to a given business process. Be precise: not every regulation applies to every process. A thorough analysis requires identifying both relevant AND irrelevant passages accurately. First, think step by step about how the regulatory text connects to the process, and then classify its relevance.
The relevant chunks will be used for compliance checking, meaning that the process will be evaluated against these chunks to verify if it fulfills the obligations or requirements described in them.
So the relevance is an applicability of the regulatory text to the process, not whether the process is compliant with it.

## Relevance Criteria

A passage IS relevant ("yes") if:
- It describes an obligation or requirement that directly governs an activity within the process.
- It regulates specific inputs, outputs, data, roles, or controls that the process handles.
- It creates a compliance duty that the process must address (e.g., reporting, documentation, approval).
- It describes a requirement that SHOULD apply to the process but is currently not reflected in it. A missing control, check, or step is a compliance gap — this makes the passage MORE relevant, not less.

A passage is partially relevant ("partially") if:
- The regulation is relevant but the process is not the primary owner.
  - The process handles only one aspect of a broader obligation (e.g., it collects data
    but another process handles the required reporting).
  - The regulation applies to a shared resource the process uses (e.g., IT infrastructure,
    customer data) but is primarily governed elsewhere.
  - The process contributes to compliance but is not solely responsible.
  -The regulatory text passage is relevant event if it contradicts the process.

A passage is NOT relevant ("no") if:
- It only shares a general topic or domain with the process but does not impose any obligation on it (e.g., a regulation about credit risk is not relevant to an HR onboarding process just because both exist in a bank).
- It applies to a different organizational function, lifecycle stage, or business area.
- The only connection is that both texts mention the same broad industry or entity type.


When uncertain: Briefly state why the connection is unclear. If you can articulate a specific, plausible obligation the process would need to fulfill, classify as "yes". If you cannot name a concrete link, classify as "no".

## Output Format
Respond strictly as JSON:
{{
  "reasoning": "<2-3 sentences: name the specific obligation in the regulation AND the specific process activity it connects to, or explain why no concrete link exists>",
  "relevance": "<yes or partially or no>"
}}

---

# Input

Regulatory text:
{chunk}

Process:
{process}
"""




relevance_evaluator_prompt = """
# Instructions

You are a meta-evaluator assessing the quality of a relevance classification - whether the regulatory text is relevant (applicable) to the process.
Your job is to evaluate the  system's output on multiple dimensions and produce a structured quality score.

## Context

You will receive:
1. The **regulatory text passage** that was analyzed
2. The **business process** description
3. The ** system's output** (a JSON with "reasoning" and "classification" fields)

Read and understan the  input.

Your job is to verify:
- Is the relevance classification assignment (yes/partially/no/) correct given the rule and process and explanation provided?
- Is there any inconsistency in the classification and classification?

The regulatory text passage is relevant event if it contradicts the process.

# Input

## Regulatory Text Passage
{rule}

## Business Process
{full_process}

##  System Output to evaluate
Classification: {category}
Explanation: {explanation}

## Output Format

Respond strictly as JSON:
{{

    "classification_correctness_justification": "<1-2 sentences explaining your evaluation of the classification correctness>"
    "classification_correctness": <true or false>,
    "final_relevance_verdict": "<yes or partially or no, if you would change the original classification based on your evaluation>"
}}

---
"""


In [ ]:
document_evaluation_prompt = """
# Instructions

You are an expert in regulatory compliance analysis. Analyze whether a regulatory text passage is relevant to a given business process. Be precise: not every regulation applies to every process. A thorough analysis requires identifying both relevant AND irrelevant passages accurately.  The relevant chunks will be used for compliance checking, meaning that the process will be evaluated against these chunks to verify if it fulfills the obligations or requirements described in them. First, think step by step about how the regulatory text connects to the process, and then classify its relevance.

## Relevance Criteria

A passage IS relevant ("yes") if:
- It describes an obligation or requirement that directly governs an activity within the process.
- It regulates specific inputs, outputs, data, roles, or controls that the process handles.
- It creates a compliance duty that the process must address (e.g., reporting, documentation, approval).
- It describes a requirement that SHOULD apply to the process but is currently not reflected in it. A missing control, check, or step is a compliance gap — this makes the passage MORE relevant, not less.

A passage is partially relevant ("partially") if:
- The regulation is relevant but the process is not the primary owner.
  - The process handles only one aspect of a broader obligation (e.g., it collects data
    but another process handles the required reporting).
  - The regulation applies to a shared resource the process uses (e.g., IT infrastructure,
    customer data) but is primarily governed elsewhere.
  - The process contributes to compliance but is not solely responsible.

A passage is NOT relevant ("no") if:
- It only shares a general topic or domain with the process but does not impose any obligation on it (e.g., a regulation about credit risk is not relevant to an HR onboarding process just because both exist in a bank).
- It applies to a different organizational function, lifecycle stage, or business area.
- The only connection is that both texts mention the same broad industry or entity type.

When uncertain: Briefly state why the connection is unclear. If you can articulate a specific, plausible obligation the process would need to fulfill, classify as "yes". If you cannot name a concrete link, classify as "no".

## Output Format
Respond strictly as JSON:
{{
  "reasoning": "<2-3 sentences: name the specific obligation in the regulation AND the specific process activity it connects to, or explain why no concrete link exists>",
  "relevance": "<yes or partially or no >"
}}

---

# Input

Regulatory text:
{chunk}

Process:
{process}
"""

In [ ]:
start = time.time()

df_relevance_to_process = run_step(
    compliance_rules, process, client, document_evaluation_prompt,
    step_name="1.0_relevance_to_process"
)
timings["relevance_to_process"] = time.time() - start
#TODO: MACHT PARTIaLLY überhaupt sinn? ja, nach  (NUR EINEM ) Experiment, dass wie ein Weg Recall zu erhöhen
# Geprüft auf reliability - die outputs sind meistens gleich

In [ ]:
df_relevance_to_process

#erkenntniss: wieso 4o zeigt mehr 'no' als 4o-mini

In [ ]:
relevant_doc_chunk_list = [
    (row['chunk_id'], row['chunk_text'])
    for _, row in df_relevance_to_process.iterrows()
    if row['relevance'] != 'no'
]

print(f"Number of relevant to process rules: {len(relevant_doc_chunk_list)} of {len(df_relevance_to_process)}")

# Compliance Analysis

In [ ]:
extraction_prompt_3 = """
# Role
You are a compliance analysis extraction expert. You receive a compliance analysis that was
performed between a process and a compliance rule. Your task is to:
1. Extract the core compliance rule aspect in plain, user-friendly language.
2. Categorize the analysis result.
3. Extract the EXACT segment(s) from the process that were analyzed and deemed important for the compliance determination.

# Input
COMPLIANCE ANALYSIS:
{compliance_analysis}

# Task

## Step 1: Extract the Rule Aspect
Read the analysis and identify the specific compliance rule that was checked.
Distill it down to its core requirement — the key "must/must not" condition
that drove the compliance decision.

### Rule extraction rules:
- Express the rule in plain, simple language a non-expert can understand immediately.
- Keep it to 1 short sentence maximum.
- Focus ONLY on the specific aspect of the rule that was relevant to this compliance decision, not the full rule.
- Strip away legal jargon, statute numbers, and technical terminology.
- Use active voice (e.g., "Companies must verify merchant identity before granting access" NOT "It is required that identity verification be conducted prior to access being granted").
- Do NOT include the compliance result — only describe WHAT the rule requires.

Example:
Rule in analysis: "All payment service providers shall conduct a comprehensive identity verification of each merchant prior to granting access to payment processing services."
Output: "Every merchant must be fully identity-verified before they can process payments."

## Step 2: Identify the Process Segment
Read the analysis carefully and extract the EXACT text segment(s) from the process
that the analysis referenced, quoted, or relied upon to reach its conclusion.

### Extraction rules:
- Extract the process text AS IT APPEARS in the analysis (verbatim quotes or close paraphrases of process steps).
- If the analysis references multiple process segments, extract ALL of them.
- Extract short, the most important part of the process for the final decision.
- If the analysis states the process does NOT mention something (i.e., no relevant segment exists), return "NO RELEVANT SEGMENT FOUND" as the extracted text.
- Do NOT extract the rule text — only extract process-side information.
- Do NOT extract the analyst's commentary — only extract the actual process content being discussed.

Example:
Input:
COMPLIANCE ANALYSIS: "...fails to ensure comprehensive verification for each merchant, which contradicts the compliance rule..."
Rule: "All payment service providers shall conduct a comprehensive identity verification of each merchant prior to granting access to payment processing services."
Original process text: "Beneficial ownership verification is performed only when flagged by the automated risk scoring tool, not for all merchants."
Output:
extracted_process_segment: "Beneficial ownership verification  not for all merchants"

## Step 3: Classify the Result
Classify into exactly ONE category:

### NOT ADDRESSED (Insufficient Evidence)
The analysis conclusion is PRIMARILY based on the ABSENCE of information —
the process simply does not mention or cover the rule's topic.

### NON-COMPLIANT
The analysis identified a genuine conflict, contradiction, or violation where
the process ACTIVELY ADDRESSES the rule's subject area but fails to meet
its specific requirements.

### COMPLIANT
The analysis found positive evidence that the process meets the rule's requirements.

### Key test for NON-COMPLIANT vs NOT ADDRESSED:
- Process does something wrong/insufficient in the same area → NON-COMPLIANT
- Process simply never talks about this area → NOT ADDRESSED

# Output Format in JSON ONLY:
{{
  "rule_aspect": "<Short, plain-language summary of the key rule requirement that was checked>",
  "category": "NOT ADDRESSED | NON-COMPLIANT | COMPLIANT",
  "extracted_process_segment": ["most important process segment as an evidence of the compliance result"],
  "reasoning": "<one sentence explaining why this category was chosen>",
  "short_evidence": "<Very short, ONLY WHAT was not addressed/violated/compliant. in bullet point>"
}}

"""

In [ ]:
prompt_compliance_report = """

# Role and Task
You are an expert Compliance Analyst. Your task is to analyze whether
a given process is compliant with a specific compliance rule.

You will work through this in three structured phases.

# Input
COMPLIANCE RULE:
{chunk}

PROCESS:
{process}

# Analysis Instructions — Follow these three phases in order.

## Phase 1: Understand the Rule
- Identify the core requirement of the compliance rule.
- What specific action, control, or condition does it mandate or prohibit?
- What type of activity, domain, or process does this rule govern?

## Phase 2: Assess Relevance
Before checking compliance, determine whether this rule is logically
relevant to the given process.

Ask yourself:
- Does this process operate in the same domain or activity area
  that the rule governs?
- Does the process handle decisions, data, tasks, or interactions
  that fall within the rule's scope?
- Would a compliance officer reasonably expect THIS specific process
  to address this rule?

State your relevance conclusion clearly:
- If the rule governs a domain/activity that this process does not
  touch at all → state that the rule is NOT RELEVANT to this process
  and stop your analysis here.
- If the rule IS relevant → proceed to Phase 3.

## Phase 3: Assess Compliance
Now that you have established the rule is relevant, check whether
the extracted process segment satisfies it.

## Output
Provide your full reasoning across all three phases. Be specific,
reference exact parts of the rule and the process, and make your
relevance and compliance conclusions explicit and unambiguous.
"""

In [ ]:
#the boundary between "the process should cover this but doesn't" vs. "this rule simply isn't relevant to this process" is inherently subjective, and LLMs tend to default to one or the other without clear guidance.
# the process doesn't mention it, but logically it should. That's actually a non-compliance by omission — and right now your prompt tells the LLM to classify omissions as NOT ADDRESSED.

In [ ]:
df_relevance_to_process = df_relevance_to_process[df_relevance_to_process['relevance'] != 'no'].reset_index(drop=True)

In [ ]:
df_relevance_to_process

In [ ]:
start = time.time()

df_compliance_comprehensive_report_process = evaluate_chunk_df_no_json(
    df_relevance_to_process,
    "chunk_text",
    "chunk_id",
    "process",
    llm_client=client,
    evaluation_prompt_template=prompt_compliance_report
)

timings["compliance_analysis"] = time.time() - start

In [ ]:
display_results_general(df_compliance_comprehensive_report_process)

In [ ]:
df_compliance_comprehensive_report_process

In [ ]:
start = time.time()

df_compliance_extraction_table_report = evaluate_chunk_df_classify_compliance(df_compliance_comprehensive_report_process, 'chunk_text', 'process', 'compliance_report', client, extraction_prompt_3)

#df_compliance_extraction_table_report.to_excel("Rel_exp/df_compliance_extraction_table_report_only_this_step_5.xlsx", index=False)
timings["compliance_table_extraction_report"] = time.time() - start

In [ ]:
df_compliance_extraction_table_report

In [ ]:
"""
from collections import Counter

# --- 5 Durchläufe ---
all_runs = []
for i in range(5):
    print(f"Durchlauf {i+1}/5 ...")
    df_run = evaluate_chunk_df_classify_compliance(
        df_compliance_comprehensive_report_process,
        'chunk_text', 'process', 'compliance_report',
        client, extraction_prompt_3
    )
    df_run = df_run.copy().reset_index(drop=True)
    df_run['_run'] = i + 1
    df_run['_row_idx'] = df_run.index
    all_runs.append(df_run)

df_all = pd.concat(all_runs, ignore_index=True)

# --- Verteilung + Finale Tabelle ---
distribution_rows = []
final_rows = []

for row_idx, grp in df_all.groupby('_row_idx'):
    counts = Counter(grp['category'])
    ranked = counts.most_common()
    most_common_cat = ranked[0][0]
    most_common_cnt = ranked[0][1]

    dist_row = {
        'chunk_id': grp['chunk_id'].iloc[0],
        'top_category': most_common_cat,
        'top_count': most_common_cnt,
        'confidence': most_common_cnt / 5,
        'second_category': ranked[1][0] if len(ranked) > 1 else None,
        'second_count': ranked[1][1] if len(ranked) > 1 else None,
    }
    distribution_rows.append(dist_row)

    match = grp[grp['category'] == most_common_cat].iloc[0]
    final_rows.append(match.drop(['_run', '_row_idx']))

df_distribution = pd.DataFrame(distribution_rows)
df_final = pd.DataFrame(final_rows).reset_index(drop=True)

print("\n=== Verteilung der Categories über 5 Durchläufe ===")
display(df_distribution)
print(f"\n=== Finale Tabelle ({len(df_final)} Zeilen) ===")
display(df_final)

df_compliance_extraction_table_report = df_final

#### ERKENNTINS _ DIE VARIABLILTY IST ZWAR DA; ABER VIEL KLEINER ALS WENN MAN GESAMT TEXT AUCH AUSGIBT

"""

In [ ]:
extraction_prompt_3

In [ ]:
##The violation_type field distinguishes between active violations and omissions within the NON-COMPLIANT category. This gives you useful downstream data: you can prioritize active violations over omissions, or present them differently in reports

In [ ]:
print(prompt_compliance_report)

In [ ]:
df_compliance_extraction_table_report
# Erkenntnis, not addressed kann hier besser evaluated wegen eventuel besseren Kontext?

# Judge

In [ ]:
# LLM Judge Prompt - Final Quality Check
judge_prompt = """
You are a Quality Assurance Judge reviewing compliance categorization results. Your task is to verify the correctness and consistency of the structured compliance assessment.

You will be provided with:
1. The original compliance rule
2. The TARGET process sentence that was evaluated
3. The full context (surrounding sentences)
4. The structured categorization result

Your job is to verify:
- Is the category assignment (COMPLIANT/NON-COMPLIANT/NOT ADDRESSED) correct given the rule and process and explanation provided?
- Is the explanation accurate and does it use terminology from the rule?
- Is there any inconsistency in the categorization?

The category "NOT ADDRESSED" should be used when the analysis indicates that the process step is missing required information or no mentions of contraints in the process.

INPUTS:

COMPLIANCE RULE:
{rule}

TARGET PROCESS SENTENCE (being evaluated):
{process_sentence}

FULL CONTEXT:
{full_process}
Structured Output from Compliance Categorization:
Explanation: {explanation}
Category: {category}


OUTPUT FORMAT:
Provide your judgment as valid JSON:
{{
  "is_category_correct": true/false,
  "issues_found": ["list any issues or inconsistencies", "empty list if none"],
  "corrected_category": "corrected category if needed, otherwise same",
  "corrected_explanation": "corrected explanation if needed, otherwise same",
  "judge_reasoning": "Brief explanation of your decision (1-2 sentences)"
}}

"""

In [ ]:
#'Es gibt aber auch die Position, dass das gleiche Modell als Judge zulässig ist, wenn die Bewertungsaufgabe klar anders ist als die eigentliche Generationsaufgabe und die Bewertungsqualität separat validiert wurde.'

In [ ]:
start = time.time()

df_compliance_category_table_report_final= evaluate_chunk_df_judge_compliance(
    df_compliance_extraction_table_report, 'chunk_text','extracted_process_segment', 'reasoning', 'category',  process, client, judge_prompt)

timings["compliance_judge"] = time.time() - start

In [ ]:
df_compliance_category_table_report_final
#chunk id verloren

In [ ]:
df_compliance_category_table_report_final[['chunk_id','category', 'corrected_category']]

In [ ]:
timings

In [ ]:
#####AMBIGUITY HANDLING
####You're right — the prompt is over-flagging ambiguity. The core issue is that your detection criteria are too broad, so terms that are contextually clear or industry-standard get treated the same as genuinely unresolvable vagueness.

prompt_ambiguity = """
You are a compliance analysis expert. Your task is to analyze regulatory rules, identify ambiguous language, map ambiguous terms to available evidence, and determine compliance outcomes through logical reasoning.

## Input
You will receive a table with the following columns:
- Rule: {rule}
- Mapped Process Segment: {process_sentence}
-Reasoning: {explanation}
Categry: {category}

## Your Task (follow these steps in order)

### Step 1 — Ambiguity Detection
Examine each rule and identify terms or phrases that are vague, or open to interpretation. These include but are not limited to:
- Time-related ambiguity: "appropriate time", "timely manner", "without undue delay", "promptly", "reasonable period"
- Quantity ambiguity: "sufficient", "adequate", "minimal", "reasonable amount"
- Quality ambiguity: "appropriate", "suitable", "proper", "satisfactory", "acceptable"
- Scope ambiguity: "relevant", "necessary", "applicable", "as needed"

Rules are considered ambiguous if they:
- Use vague temporal or qualitative terms such as “in a timely manner”, “as soon as possible”, “promptly”, “without undue delay”, “within a reasonable time”, “as a rule”, “where appropriate”, “whenever possible”, etc., without a concrete time frame, measurable threshold, or objective criterion.
- Are open to multiple reasonable interpretations in practice (for example, different stakeholders could reasonably apply the rule differently).
- Lack quantifiable criteria (for example, no explicit time limit, no defined exception, no clear metric).
- vague or imprecise enough to cause compliance interpretation issues

Rules are not ambiguous if they:
- Specify concrete time frames (for example, “within 5 business days”), measurable thresholds (for example, “within 24 hours of receipt”), or objective criteria (for example, “by the last business day of the month”).
- Are clearly constrained by context, law, or policy such that interpretation is limited and predictable.

If a rule contains NO ambiguous terms (i.e., it is fully specific and measurable), mark it as "Clear Rule — No Assumption Needed" and skip further analysis for that rule.

Only proceed with Steps 2–4 for rules that contain ambiguous terms.

### Step 2 — Evidence Mapping
For each ambiguous rule, examine the provided evidence and identify which specific evidence items relate to the ambiguous term. Extract the concrete, measurable value from the evidence. For example, if the rule says "Claims must be processed within an appropriate time" and the evidence states "Claims were processed during 20 days", then the evidence value for "appropriate time" is "20 days".

### Step 3 — Assumption Formation
Based on the compliance result provided, form a logical assumption that bridges the ambiguous term to the evidence.

Use this reasoning pattern:
- IF compliance result = "Compliant" → THEN the evidence value IS an acceptable interpretation of the ambiguous term
- IF compliance result = "Non-Compliant" → THEN the evidence value IS NOT an acceptable interpretation of the ambiguous term

State the assumption as a clear, reusable interpretation rule.

### Step 4 — Output
For each rule that contains ambiguity, return a structured result:

Provide your judgment as valid JSON:
{{
  "ambiguous_term:": ["the vague phrase from the rule"]
  "mapped_evidence": ["the specific, measurable fact from evidence"],
  "assumtion": "assumtion needed to evaluate compliance given the ambiguity",
  "compliance_category": "COMPLIANT or NON-COMPLIANT",
  "reasoning": "Brief explanation of your decision (1-2 sentences)"
}}


## Example

Input:
Rule: "Claims must be processed within an appropriate time"
Mapped Process Segment: "Claims were processed during 20 days"
Compliance Result: "Compliant"

Output:
Ambiguous Term: "appropriate time"
Mapped Evidence: "20 days"
Assumption: "Processing claims within 20 days is considered an acceptable interpretation of 'appropriate time' for the Claims Handling process"
Compliance Category: "COMPLIANT"
Reasoning: "Since the compliance result is 'Compliant' and the evidence shows claims were processed within 20 days, we can infer that in this context, 'appropriate time' is interpreted as 20 days or less. This assumption allows us to bridge the ambiguity in the rule with the concrete evidence provided."


## Important Guidelines
- Only flag genuinely ambiguous terms. Do not flag specific, measurable language (e.g., "within 30 days" is NOT ambiguous).
- One rule may contain multiple ambiguous terms — analyze each separately.
- Assumptions must be scoped to the specific process. The same ambiguous term may have different acceptable thresholds in different processes.
- If no evidence maps to the ambiguous term, state: "No matching evidence found — assumption cannot be formed."
- Be precise in your reasoning. Do not infer beyond what the evidence and compliance result support.
"""


In [ ]:
prompt_ambiguity = """
You are a compliance analysis expert. Your task is to determine whether evaluating a rule against its evidence REQUIRES making an assumption — and if so, to state that assumption explicitly.

## Input
You will receive a table with the following columns:
- Rule: {rule}
- Mapped Process Segment: {process_sentence}
- Reasoning: {explanation}
- Category: {category}

## Your Task (follow these steps in order)

### Step 1 — Compliance Decision Test
For each rule, ask yourself ONE question:

  "Can I determine compliance or non-compliance using ONLY the rule text and the mapped evidence, without making any interpretive assumption?"

- If YES → mark as "No Assumption Needed" and skip to the next rule. Do NOT analyze further.
- If NO → proceed to Step 2.

A rule does NOT need an assumption when:
- It contains specific, measurable criteria (e.g., "within 30 days", "at least 3 reviewers", "by end of quarter")
- The vague-sounding term is actually constrained by industry norms, legal definitions, or the surrounding rule context to a single reasonable interpretation
- The evidence directly and unambiguously satisfies or violates the rule regardless of how you interpret the flexible term
- The term is standard regulatory language with a well-understood practical meaning in the given domain (e.g., "applicable laws" in a legal compliance context is not ambiguous — it means the laws that apply)

A rule DOES need an assumption when:
- The rule uses a term with no measurable threshold AND the compliance verdict changes depending on how you interpret that term
- Two reasonable professionals could look at the same evidence and reach opposite compliance conclusions because the rule is underspecified
- You must invent or assume a benchmark, deadline, quantity, or standard that the rule does not provide in order to compare against the evidence

### Step 2 — Identify the Decision-Blocking Term
Pinpoint the SPECIFIC term or phrase that forces you to make an assumption. This is the term where, if it were replaced with a concrete value, the compliance decision would become straightforward.

Do NOT flag:
- Terms that sound vague in isolation but are clear in context
- Multiple terms from the same rule if only one actually blocks the decision
- Adjectives or qualifiers that don't affect the compliance outcome

### Step 3 — Evidence Mapping
Extract the concrete, measurable value from the mapped evidence that corresponds to the ambiguous term.

If no evidence maps to the ambiguous term, state: "No matching evidence found — assumption cannot be formed." and stop.


### Step 5 — Output
For each rule that REQUIRES an assumption, return:

{{
  "ambiguous_term": "the specific term that blocks the compliance decision",
  "mapped_evidence": "the concrete value from the evidence",
  "assumption": "the interpretive bridge needed to reach a compliance verdict",
  "new_category": "new compliance category resul based on the assumption"

}}

For rules that do NOT require an assumption, return:

{{
  "ambiguous_term": "None",
  "mapped_evidence": "N/A",
  "assumption": "No Assumption Needed",
  "new_category": "COMPLIANT or NON-COMPLIANT or NOT ADDRESSED",
}}

## Examples

### Example 1 — Assumption IS Needed
Rule: "Claims must be processed within an appropriate time"
Mapped Process Segment: "Claims were processed during 20 days"
Category: "NON-COMPLIANT"

Output:
{{
  "ambiguous_term": "appropriate time",
  "mapped_evidence": "20 days",
  "assumption": "20 days is 'appropriate time',
  "new_category": "COMPLIANT",

}}

### Example 2 — Assumption is NOT Needed
Rule: "All claims must be acknowledged within 5 business days of receipt"
Mapped Process Segment: "Claims were acknowledged on day 3 after receipt"
Category: "COMPLIANT"

Output:
{{
  "ambiguous_term": "None",
  "mapped_evidence": "N/A",
  "assumption": "No Assumption Needed",
  "new_category": "COMPLIANT"
}}

### Example 3 — Assumption is NOT Needed (contextually clear term)
Rule: "The insurer must comply with all applicable data protection regulations"
Mapped Process Segment: "The insurer follows GDPR and local data privacy laws"
Category: "COMPLIANT"

Output:
{{
  "ambiguous_term": "None",
  "mapped_evidence": "N/A",
  "assumption": "No Assumption Needed",
  "new_category": "COMPLIANT"
}}

## Critical Guardrails
- Your DEFAULT should be "No Assumption Needed." Only flag ambiguity when it genuinely blocks the compliance decision.
- If you are unsure whether an assumption is needed, apply the TWO-PROFESSIONAL test: would two reasonable compliance professionals, given the same rule and evidence, potentially reach OPPOSITE conclusions? If not, no assumption is needed.
- Do not flag industry-standard regulatory language as ambiguous unless it truly lacks a commonly understood meaning in the specific domain.
- One rule may occasionally have multiple decision-blocking terms, but this is rare. Most rules have zero or one.
"""

In [ ]:
df_compliance_category_table_report_final

In [ ]:
start = time.time()

df_compliance_amb_assumtions= evaluate_chunk_df_judge_compliance(
    df_compliance_category_table_report_final, 'rule_aspect','extracted_process_segment', 'corrected_explanation', 'category',  process, client, prompt_ambiguity)

timings["compliance_assumtion"] = time.time() - start

In [ ]:
df_compliance_amb_assumtions

In [ ]:
df_compliance_category_table_report_final = df_compliance_amb_assumtions

In [ ]:
#df_compliance_category_table_report_final.to_excel("semi_final_output.xlsx", index=False)


In [ ]:
#df_compliance_category_table_report_final = pd.read_excel(
#    "semi_final_output.xlsx" )

In [ ]:
df_compliance_category_table_report_final

'# Exact Sentence Matching

In [ ]:
# pip install sentence-transformers pandas
# Angepasst: TOP 3 ähnlichsten Segmente (KEIN Threshold!)

# Model laden
model = SentenceTransformer('all-MiniLM-L6-v2')


# ANWENDUNG AUF DEINEN DATAFRAME
df_compliance_category_table_report_final['top_similar_sentences'] = df_compliance_category_table_report_final.apply(
    lambda row: find_top_similar_sentences(
        row['extracted_process_segment'],  # Query
        row.get('original_text', row.get('process_sentence', ''))  # Passe Spaltenname an!
    ), axis=1
)

# Ergebnis
print("Top-3 ähnlichste Sätze pro Query hinzugefügt!")
print(df_compliance_category_table_report_final[['extracted_process_segment', 'top_similar_sentences']].head())

# Mit Match-Count
df_compliance_category_table_report_final['top_match_count'] = df_compliance_category_table_report_final['top_similar_sentences'].apply(len)


In [ ]:
df_compliance_category_table_report_final

In [ ]:
process_str =str(process)

In [ ]:
type(process)

In [ ]:
df_compliance_category_table_report_final["process_id"] = (
    df_compliance_category_table_report_final["top_similar_sentences"]
    .apply(lambda lst: min(
        (process_str.find(x) for x in lst if isinstance(x, str) and process_str.find(x) != -1),
        default=-1
    ))
)

In [ ]:
df_compliance_category_table_report_final

In [ ]:
df_compliance_category_table_report_final = df_compliance_category_table_report_final.rename(
    columns={
        'rule_aspect': 'easy_rule',
        'extracted_process_segment': 'extracted_process_segments_from_output',
        'top_similar_sentences': 'extracted_process_segment',
        'category': 'initial_category',
        'new_category': 'category',
    }
)

In [ ]:
df_compliance_category_table_report_final.to_excel('final_2.xlsx', index= False)

In [ ]:


#df_relevance_to_process.to_excel(
#    "Rel_exp/df_relevance_to_process_5.xlsx",
#    index=False
#)
#df_relevance_to_process.to_excel( "Rel_exp/df_relevance_to_process_5.xlsx", index=False)

#df_chunk_classification.to_excel(
#    "Rel_exp/df_chunk_classification_5.xlsx", index=False )
#df_compliance_comprehensive_report_process.to_excel("Rel_exp/df_compliance_comprehensive_report_process_5.xlsx", index=False)
#df_compliance_extraction_table_report.to_excel("Rel_exp/df_compliance_extraction_table_report_5.xlsx", index=False)
#df_compliance_category_table_report_final.to_excel("Rel_exp/df_compliance_category_table_report_final_5.xlsx",  index=False)


In [ ]:
# Es gibt mehrere Bereiche, in denen ein größeres Modell paradoxerweise schlechter abschneiden kann:
#Einfache, klar definierte Aufgaben: Bei simplen Extraktions-, Klassifikations- oder Formatierungsaufgaben neigen größere #Modelle manchmal dazu, zu „überdenken" – sie fügen ungewollte Nuancen, Caveats oder zusätzliche Informationen hinzu, wo #eine direkte Antwort gefragt war. Ein kleineres Modell folgt der Anweisung oft buchstabengetreuer.
#Strukturierte Ausgaben (JSON, Code-Templates): GPT-4o neigt eher dazu, von einem streng vorgegebenen Format abzuweichen – #etwa durch zusätzliche Erklärungen, Markdown-Formatierung oder kleine „hilfreiche" Ergänzungen, die dann das Parsing #brechen. GPT-4o-mini ist in solchen Fällen oft zuverlässiger.
#